# Model Latency Test

Test latency characteristics for a model hosted in Azure AI Foundry.

**Metrics captured:** Time to First Token (TTFT), Total Response Time, Tokens/sec

## 1. Setup

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from utils import get_client, get_model_name, get_openai_client, run_latency_test, run_responses_latency_test

## 2. Configure Model Endpoint

Reads from `.env` by default. Use `responses` for Azure OpenAI / Foundry OpenAI base URLs such as `*.openai.azure.com` or endpoints ending in `/openai/v1`. Use `inference` for Azure AI Inference endpoints such as `*.services.ai.azure.com`.

In [ ]:
# Use "responses" for Azure OpenAI / Foundry OpenAI base URLs.
# Use "inference" for Azure AI Inference endpoints.
API_TYPE = "responses"

# Override the endpoint inline if needed:
# endpoint = "https://your-resource.openai.azure.com"
# endpoint = "https://your-endpoint.services.ai.azure.com"
endpoint = None

if API_TYPE == "responses":
    client = get_openai_client(endpoint=endpoint)
    run_test = run_responses_latency_test
elif API_TYPE == "inference":
    client = get_client(endpoint=endpoint)
    run_test = run_latency_test
else:
    raise ValueError("API_TYPE must be 'responses' or 'inference'")

model = get_model_name()

print(f"API type: {API_TYPE}")
print(f"Model: {model}")

API type: inference
Model: gpt-4o-mini


## 3. Define Test Prompts

Add, remove, or modify prompts to vary the test. A mix of short and long expected outputs is useful.

In [3]:
prompts = [
    "What is the capital of France?",
    "Explain quantum computing in 3 sentences.",
    "Write a Python function that checks if a string is a palindrome.",
    "Summarize the key ideas of machine learning in a short paragraph.",
]

## 4. Run Latency Measurements

In [6]:
results = run_test(client, model, prompts)
print(f"Completed {len(results)} measurements")

Completed 4 measurements


## 5. Results

In [ ]:
df = pd.DataFrame(results)
display_cols = ["prompt_index", "prompt", "ttft", "total_time", "tokens_per_second", "completion_tokens"]
df[display_cols]

In [ ]:
print("=== Summary ===")
print(f"Avg TTFT:           {df['ttft'].mean():.4f}s")
print(f"Avg Total Time:     {df['total_time'].mean():.4f}s")
print(f"Avg Tokens/sec:     {df['tokens_per_second'].mean():.2f}")
print(f"Total Tokens:       {df['completion_tokens'].sum()}")

## 6. Visualize (Optional)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(df["prompt_index"], df["ttft"])
axes[0].set_title("Time to First Token (s)")
axes[0].set_xlabel("Prompt")

axes[1].bar(df["prompt_index"], df["total_time"])
axes[1].set_title("Total Response Time (s)")
axes[1].set_xlabel("Prompt")

axes[2].bar(df["prompt_index"], df["tokens_per_second"])
axes[2].set_title("Tokens per Second")
axes[2].set_xlabel("Prompt")

plt.tight_layout()
plt.show()